In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ----------------------------
# 1) Robust time parsing
# ----------------------------
def parse_time_col(s: pd.Series) -> pd.Series:
    """
    Handles either:
      - numeric unix timestamps (seconds OR milliseconds)
      - string/ISO timestamps
    """
    if pd.api.types.is_numeric_dtype(s):
        med = float(pd.Series(s).dropna().median())
        unit = "ms" if med > 1e12 else "s"
        return pd.to_datetime(s, unit=unit, errors="coerce")
    return pd.to_datetime(s, errors="coerce")


# ----------------------------
# 2) Load all sections for a course-term
# ----------------------------
def load_course(quarter, course_id, base_path="data", dept="cogs"):
    """
    Loads all section CSVs for a given course in a given quarter
    and returns a section-level event-log dataframe.
    """

    df = pd.read_csv(f"{base_path}/{quarter}/{course_id}.csv")

    # standardize column names
    df = df.rename(columns={
        "waitlist": "waitlisted",
        "enrolled_ct": "enrolled"
    })

    # parse time robustly
    df["datetime"] = parse_time_col(df["time"])


    # attach identity
    df["course"] = course_id.replace("_", " ")
    df["dept"] = dept
    df["quarter"] = quarter


    return df


# ----------------------------
# 3) Aggregate sections -> course-level time series
# ----------------------------
def aggregate_to_course_ts(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Sums enrolled/available/waitlisted/total across sections per timestamp.
    Returns a course-level time series.
    """
    required = {"quarter","dept","course","datetime","enrolled","available","waitlisted","total"}
    missing = required - set(df_raw.columns)
    if missing:
        raise ValueError(f"aggregate_to_course_ts missing columns: {missing}")

    course_ts = (
        df_raw
        .groupby(["quarter", "dept", "course", "datetime"], as_index=False)
        .agg({
            "enrolled": "sum",
            "available": "sum",
            "waitlisted": "sum",
            "total": "sum"
        })
        .sort_values("datetime")
        .reset_index(drop=True)
    )
    return course_ts


# ----------------------------
# 4) 12-hour bin fill rates (0-72h) from a fixed anchor time
# ----------------------------
def _prep_time(df):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    return df.sort_values("datetime").reset_index(drop=True)

def _value_nearest(df, target_time, col):
    idx = (df["datetime"] - target_time).abs().idxmin()
    return float(df.loc[idx, col])

def fill_rates_by_12h_bins(course_ts: pd.DataFrame,
                           start_time="2022-05-20 08:00:00",
                           tz=None):
    df = _prep_time(course_ts)

    t0 = pd.to_datetime(start_time)
    if tz is not None:
        if df["datetime"].dt.tz is None:
            df["datetime"] = df["datetime"].dt.tz_localize(tz)
        if t0.tzinfo is None:
            t0 = t0.tz_localize(tz)

    t_end = t0 + pd.Timedelta(hours=72)

    # small buffer so nearest-matching works at edges
    window_df = df[
        (df["datetime"] >= t0 - pd.Timedelta(hours=2)) &
        (df["datetime"] <= t_end + pd.Timedelta(hours=2))
    ].copy()

    if len(window_df) < 2:
        return None

    boundaries = [t0 + pd.Timedelta(hours=h) for h in (0, 12, 24, 36, 48, 60, 72)]
    enrolled_at = [_value_nearest(window_df, t, "enrolled") for t in boundaries]

    rates = {}
    labels = [(0,12),(12,24),(24,36),(36,48),(48,60),(60,72)]
    for i, (a,b) in enumerate(labels):
        rates[f"fill_rate_{a}to{b}h"] = (enrolled_at[i+1] - enrolled_at[i]) / 12.0

    return rates


# ----------------------------
# 5) Build final dataset for a list of courses in a quarter
# ----------------------------
def value_at_time_nearest(df, target_time, col="waitlisted"):
    idx = (df["datetime"] - target_time).abs().idxmin()
    return float(df.loc[idx, col])

def build_dataset_for_courses(quarter, course_ids, base_path="data", dept="cogs",
                              start_time="2022-05-20 08:00:00",
                              end_date="2022-06-03",   # YYYY-MM-DD
                              end_hour=6,
                              tz=None):
    rows = []

    # construct end_time
    end_time = pd.to_datetime(f"{end_date} {end_hour:02d}:00:00")

    for course_id in course_ids:
        df_raw = load_course(quarter, course_id, base_path=base_path, dept=dept)
        course_ts = aggregate_to_course_ts(df_raw)

        if tz is not None:
            if course_ts["datetime"].dt.tz is None:
                course_ts["datetime"] = course_ts["datetime"].dt.tz_localize(tz)
            if end_time.tzinfo is None:
                end_time = end_time.tz_localize(tz)

        feats = fill_rates_by_12h_bins(course_ts, start_time=start_time, tz=tz)
        if feats is None:
            continue

        course_space = course_id.replace("_", " ").strip()
        feats["quarter"] = quarter
        feats["dept"] = dept
        feats["subj_course_id"] = course_space
        feats["start_time"] = start_time
        feats["end_time"] = str(end_time)

        feats["final_waitlist_count"] = value_at_time_nearest(course_ts, end_time, col="waitlisted")

        rows.append(feats)

    return pd.DataFrame(rows)



In [ ]:
import os
import pandas as pd

# 1. Define Quarter Dates (Enrollment Start & Instruction Start)

quarter_dates = {
    "Fa22": ("2022-05-20 08:00:00", "2022-09-22"),
    "Wi23": ("2022-11-07 08:00:00", "2023-01-09"),
    "Sp23": ("2023-02-18 08:00:00", "2023-04-03"),
    "Fa23": ("2023-05-26 08:00:00", "2023-09-28"),
    "Wi24": ("2023-11-14 08:00:00", "2024-01-08"),
    "Sp24": ("2024-02-17 08:00:00", "2024-04-01"),
    "Fa24": ("2023-05-24 08:00:00", "2024-09-26"),
    "Wi25": ("2023-11-12 08:00:00", "2025-01-06"),
    "Sp25": ("2024-02-15 08:00:00", "2025-03-31"),

}

# 2. Automated Builder Loop
all_quarter_dfs = []

# Iterate through both departments
for dept in ["COGS", "MATH"]:
    dept_path = os.path.join("data", dept)

    # skip if folder doesn't exist
    if not os.path.exists(dept_path):
        print(f"Skipping {dept} (folder not found)")
        continue

    # Iterate through each quarter folder inside the department (e.g., Fa22, Wi23)
    for quarter in os.listdir(dept_path):

        quarter_path = os.path.join(dept_path, quarter)

        # Check if we have dates for this quarter
        if quarter not in quarter_dates:
            print(f"Skipping {quarter} (Dates not defined in quarter_dates dictionary)")
            continue

        start_time, end_date = quarter_dates[quarter]

        # Get list of course IDs from filenames (e.g., "COGS 1.csv" -> "COGS 1")
        # We assume files are named like "COGS 1.csv"
        files = [f for f in os.listdir(quarter_path) if f.endswith(".csv")]
        course_ids = [f.replace(".csv", "") for f in files]

        print(f"Processing {dept} {quarter}: {len(course_ids)} courses found...")

        # Run the build function for this specific batch
        try:
            df_batch = build_dataset_for_courses(
                quarter=quarter,
                course_ids=course_ids,
                base_path=dept_path,   # Points to data/COGS or data/MATH
                dept=dept.lower(),     # "cogs" or "math"
                start_time=start_time,
                end_date=end_date,
                end_hour=6,
                tz=None
            )
            all_quarter_dfs.append(df_batch)
        except Exception as e:
            print(f"  Error processing {dept} {quarter}: {e}")


# 3. Merge & Save
if all_quarter_dfs:
    final_df = pd.concat(all_quarter_dfs, ignore_index=True)

    # Save the giant dataset
    final_df.to_csv("data/course_enrollment_data.csv", index=False)

    print(f"Total Observations: {len(final_df)}")
    display(final_df.head())
else:
    print("No data was generated.")

Processing COGS Wi25: 38 courses found...
Processing COGS Fa22: 31 courses found...
Processing COGS Sp23: 32 courses found...


In [ ]:
#final_df = build_dataset_for_courses(
    #quarter="Fa22",
    #course_ids=["COGS 1", "COGS 2", "COGS 3"],
    #base_path="data/COGS",
    #dept="cogs",
    #start_time="2022-05-20 08:00:00",
    #end_date="2022-09-19",  # choose your term-specific end date
    #end_hour=6,
    #tz=None
#)
#final_df


In [ ]:
final_df.to_csv("data/course_enrollment_data.csv", index=False)

print("File saved to: data/course_enrollment_data.csv")

File saved to: data/course_enrollment_data.csv
